In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

GPU available: True
GPU name: Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import os
data_path = '/content/drive/MyDrive/codebert_data'
print(os.listdir(data_path))

['validation_tokenised.json', 'train_tokenised.json', 'test_tokenised.json']


In [5]:
import json

with open('/content/drive/MyDrive/codebert_data/train_tokenised.json') as f:
    train = json.load(f)

    print("Train samples:", len(train))
    print("Keys in each sample:", train[0].keys())
    print("Input length (should be 512):", len(train[0]['input_ids']))
    print("Example label:", train[0]['label'])

Train samples: 21796
Keys in each sample: dict_keys(['input_ids', 'attention_mask', 'label'])
Input length (should be 512): 512
Example label: 0


In [6]:
!pip install -q transformers==4.44.0 scikit-learn
print("Libraries ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Libraries ready


In [7]:
import json
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
assert device.type == "cuda", "GPU not active! Redo Runtime > Change runtime type > T4 GPU"

Using device: cuda


In [15]:
DATA = '/content/drive/MyDrive/codebert_data'

def load(name):
    with open(f'{DATA}/{name}') as f:
            return json.load(f)

            train_data = load('train_tokenised.json')
            val_data   = load('validation_tokenised.json')
            test_data  = load('test_tokenised.json')

            print(f"Train: {len(train_data):,}  Val: {len(val_data):,}  Test: {len(test_data):,}")

In [16]:
import os, json

DATA = '/content/drive/MyDrive/codebert_data'

# 1. Confirm the folder and files exist
print("Files in folder:", os.listdir(DATA))

# 2. Try loading each file with a print after each
print("Loading train...")
with open(f'{DATA}/train_tokenised.json') as f:
    train_data = json.load(f)
    print("  train loaded:", len(train_data))

    print("Loading validation...")
    with open(f'{DATA}/validation_tokenised.json') as f:
        val_data = json.load(f)
        print("  val loaded:", len(val_data))

        print("Loading test...")
        with open(f'{DATA}/test_tokenised.json') as f:
            test_data = json.load(f)
            print("  test loaded:", len(test_data))

            print(f"\nDONE — Train: {len(train_data):,}  Val: {len(val_data):,}  Test: {len(test_data):,}")

Files in folder: ['validation_tokenised.json', 'train_tokenised.json', 'test_tokenised.json']
Loading train...
  train loaded: 21796
Loading validation...
  val loaded: 2732
Loading test...
  test loaded: 2731

DONE — Train: 21,796  Val: 2,732  Test: 2,731


In [25]:
class CodeDataset(Dataset):
  def __init__(self, data):
    self.data = data

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    it = self.data[idx]
    return {"input_ids": torch.tensor(it["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(it["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(it["label"], dtype=torch.long)}

BATCH_SIZE = 16
train_loader = DataLoader(CodeDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(CodeDataset(val_data), batch_size=BATCH_SIZE)
test_loader = DataLoader(CodeDataset(test_data), batch_size=BATCH_SIZE)
print("Data loaders ready. Batches per epoch:", len(train_loader))

Data loaders ready. Batches per epoch: 1363


In [26]:
model = RobertaForSequenceClassification.from_pretrained(
    "microsoft/codebert-base",
    num_labels=2   # binary: buggy (1) vs clean (0)
)
model.to(device)
print("CodeBERT loaded and moved to GPU")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CodeBERT loaded and moved to GPU


In [27]:
from torch.optim import AdamW

EPOCHS = 3          # how many times we pass through the training data
LR = 2e-5           # learning rate (standard for fine-tuning, matches your papers)

optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

print(f"Will train for {EPOCHS} epochs, {total_steps} total steps")

Will train for 3 epochs, 4089 total steps


In [29]:
import time
from sklearn.metrics import f1_score, accuracy_score

def evaluate(loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"]
            out = model(input_ids=input_ids, attention_mask=mask)
            p = torch.argmax(out.logits, dim=1).cpu().numpy()
            preds.extend(p)
            trues.extend(labels.numpy())
    return trues, preds

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    start = time.time()
    for i, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        out = model(input_ids=input_ids, attention_mask=mask, labels=labels)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if i % 200 == 0:
            print(f"  Epoch {epoch+1}, batch {i}/{len(train_loader)}, loss {loss.item():.4f}")

    # Validation after each epoch
    y_true, y_pred = evaluate(val_loader)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    mins = (time.time() - start) / 60
    print(f"Epoch {epoch+1} done in {mins:.1f} min | avg loss {total_loss/len(train_loader):.4f} | Val F1 {f1:.3f} | Val Acc {acc:.3f}")

    # Save checkpoint to Drive (so a disconnect doesn't lose progress)
    model.save_pretrained(f'{DATA}/checkpoint_epoch{epoch+1}')
    print(f"  Checkpoint saved to Drive: checkpoint_epoch{epoch+1}")

print("\nTRAINING COMPLETE")

  Epoch 1, batch 0/1363, loss 0.7240
  Epoch 1, batch 200/1363, loss 0.6637
  Epoch 1, batch 400/1363, loss 0.5744
  Epoch 1, batch 600/1363, loss 0.7073
  Epoch 1, batch 800/1363, loss 0.5889
  Epoch 1, batch 1000/1363, loss 0.5815
  Epoch 1, batch 1200/1363, loss 0.6813
Epoch 1 done in 35.3 min | avg loss 0.6391 | Val F1 0.439 | Val Acc 0.638
  Checkpoint saved to Drive: checkpoint_epoch1
  Epoch 2, batch 0/1363, loss 0.5616
  Epoch 2, batch 200/1363, loss 0.5888
  Epoch 2, batch 400/1363, loss 0.6506
  Epoch 2, batch 600/1363, loss 0.5939
  Epoch 2, batch 800/1363, loss 0.5934
  Epoch 2, batch 1000/1363, loss 0.4906
  Epoch 2, batch 1200/1363, loss 0.4625
Epoch 2 done in 35.7 min | avg loss 0.5805 | Val F1 0.510 | Val Acc 0.657
  Checkpoint saved to Drive: checkpoint_epoch2
  Epoch 3, batch 0/1363, loss 0.4712
  Epoch 3, batch 200/1363, loss 0.5096
  Epoch 3, batch 400/1363, loss 0.4406
  Epoch 3, batch 600/1363, loss 0.6042
  Epoch 3, batch 800/1363, loss 0.4472
  Epoch 3, batch 10

In [30]:
print("Evaluating on the held-out test set...")
y_true, y_pred = evaluate(test_loader)

p = precision_score(y_true, y_pred)
r = recall_score(y_true, y_pred)
f = f1_score(y_true, y_pred)
a = accuracy_score(y_true, y_pred)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("\n" + "="*50)
print("CODEBERT — FINAL TEST SET RESULTS")
print("="*50)
print(f"  Precision: {p:.3f}")
print(f"  Recall:    {r:.3f}")
print(f"  F1-score:  {f:.3f}")
print(f"  Accuracy:  {a:.3f}")
print(f"  Confusion: TP={tp} FP={fp} TN={tn} FN={fn}")
print("="*50)
print(f"Test set: {len(y_true):,} functions")

Evaluating on the held-out test set...

CODEBERT — FINAL TEST SET RESULTS
  Precision: 0.626
  Recall:    0.548
  F1-score:  0.584
  Accuracy:  0.642
  Confusion: TP=687 FP=411 TN=1066 FN=567
Test set: 2,731 functions


In [31]:
model.save_pretrained(f'{DATA}/codebert_final')
print("Final model saved to Drive: codebert_data/codebert_final")

Final model saved to Drive: codebert_data/codebert_final
